In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

HERE = "C:/Users/vishe/Downloads/Assignment/Datasets"

Sector Mapping

In [ ]:
SECTORS = {
    "01-03": "Agriculture, forestry & fishing",
    "05-39": "Production",
    "41-43": "Construction",
    "45":    "Motor trades",
    "46":    "Wholesale",
    "47":    "Retail",
    "49-53": "Transport & storage",
    "55-56": "Accommodation & food services",
    "58-63": "Information & communication",
    "64-66": "Finance & insurance",
    "68":    "Property",
    "69-75": "Professional, scientific & technical",
    "77-82": "Business administration & support",
    "84":    "Public administration & defence",
    "85":    "Education",
    "86-88": "Health",
    "90-99": "Arts, entertainment, recreation & other",
}

LAD_PREFIXES = ("E06", "E07", "E08", "E09", "W06", "S12", "N09")
CODE_RE      = re.compile(r"^[A-Z]\d{8}$")
CODE_LABEL_RE = re.compile(r"^\s*([A-Z]\d{8})\s*:\s*(.+?)\s*$")
SIC_FROM_HEADER = re.compile(r"^\s*([\d\-/,\s]+?)\s*:\s*")

print(f"{len(SECTORS)} sectors defined")

In [ ]:
def parse_sic(label) -> str | None:
    """Pull '01-03' out of a column header like 'SIC07: 01-03 : Agriculture...'"""
    if not isinstance(label, str):
        return None
    s = label.replace("SIC07:", "").strip()
    m = SIC_FROM_HEADER.match(s)
    if not m:
        return None
    sic = m.group(1).strip().replace(" ", "")
    return sic if sic in SECTORS else None

# quick sanity-check
test_headers = [
    "SIC07: 01-03 : Agriculture, forestry and fishing",
    "SIC07: 64-66 : Financial and insurance activities",
    "Total",   # should be None
    "SIC07: 99-XX : junk",  # not in SECTORS, should be None
]
for h in test_headers:
    print(repr(h), "->", parse_sic(h))

2014 to 2016


In [ ]:
ERA1_SHEETS = {
    2014: "CA_LONB_MD_NMD",
    2015: "CA_LONB_MD_NMD",
    2016: "CA_LGD_LONB_MD_NMD",
}

def _to_long(wide, year, name_col="region_name", code_col="region_code"):
    sic_cols = [c for c in wide.columns if c in SECTORS]
    id_vars  = [code_col, name_col] if {code_col, name_col} <= set(wide.columns) \
               else [c for c in wide.columns if c not in sic_cols]
    long = wide.melt(id_vars=id_vars, value_vars=sic_cols,
                     var_name="sic_codes", value_name="enterprises")
    if code_col != "region_code":
        long = long.rename(columns={code_col: "region_code", name_col: "region_name"})
    long["enterprises"]   = pd.to_numeric(long["enterprises"], errors="coerce")
    long["industry_group"] = long["sic_codes"].map(SECTORS)
    long["year"]           = year
    return long[["year","region_code","region_name","industry_group","sic_codes","enterprises"]]

def is_lad(code):
    return isinstance(code, str) and len(code) == 9 and code[:3] in LAD_PREFIXES


def read_era1(path, year):
    raw = pd.read_excel(path, sheet_name=ERA1_SHEETS[year], header=None)

    # header row detection — 'Entity Type' appears in col 5
    entity_row_idx = next(
        (r for r in range(15)
         if isinstance(raw.iat[r, 5], str) and "Entity Type" in raw.iat[r, 5]),
        None,
    )
    if entity_row_idx is None:
        raise RuntimeError(f"No 'Entity Type' header found in {path.name}")
    print(f"  {year}: entity_row_idx={entity_row_idx}")

    entity_row = raw.iloc[entity_row_idx]
    sector_row = raw.iloc[entity_row_idx + 1]

    enterprise_cols = []
    for c in range(5, raw.shape[1]):
        ent = str(entity_row.iloc[c]) if pd.notna(entity_row.iloc[c]) else ""
        if "Enterprise" not in ent:
            if enterprise_cols:
                break
            continue
        sic = parse_sic(sector_row.iloc[c])
        if sic:
            enterprise_cols.append((c, sic))

    print(f"  {year}: {len(enterprise_cols)} enterprise sector columns found")

    # geography rows
    data_start = entity_row_idx + 2
    for r in range(entity_row_idx + 2, min(entity_row_idx + 6, raw.shape[0])):
        if isinstance(raw.iat[r, 0], str) and "Parent" in raw.iat[r, 0]:
            data_start = r + 1
            break

    geo    = raw.iloc[data_start:, :5].copy()
    geo.columns = ["parent_name","parent_code","area_name","area_code","area_type"]
    geo    = geo[geo["area_code"].astype(str).str.match(CODE_RE, na=False)]

    counts = raw.iloc[data_start:, [c for c, _ in enterprise_cols]].copy()
    counts.columns = [sic for _, sic in enterprise_cols]
    counts = counts.loc[geo.index]

    return _to_long(pd.concat([geo, counts], axis=1), year,
                   name_col="area_name", code_col="area_code")

In [ ]:
era1_files = [
    (2014, "ukba01b2014.xls"),
    (2015, "ukbaa01b2015.xls"),
    (2016, "ukbaa01b2016.xls"),
]

era1_dfs = {}
for year, fname in era1_files:
    p = HERE / fname
    if not p.exists():
        print(f"  SKIP: {fname} not found")
        continue
    df = read_era1(p, year)
    df = df[df["region_code"].apply(is_lad)]
    era1_dfs[year] = df
    print(f"  {year}: {df['region_code'].nunique()} LADs, {len(df):,} rows")

if era1_dfs:
    pd.concat(era1_dfs.values()).groupby("year")["region_code"].nunique()

2017 to 2021


In [ ]:
def read_era2(path, year):
    raw = pd.read_excel(path, sheet_name="Table 1", header=None)
    sector_row = raw.iloc[5]

    sector_cols = [(c, parse_sic(sector_row.iloc[c]))
                   for c in range(2, raw.shape[1])
                   if parse_sic(sector_row.iloc[c])]

    geo = raw.iloc[6:, :2].copy()
    geo.columns = ["region_code", "region_name"]
    geo = geo[geo["region_code"].astype(str).str.match(CODE_RE, na=False)]

    counts = raw.iloc[6:, [c for c, _ in sector_cols]].copy()
    counts.columns = [sic for _, sic in sector_cols]
    counts = counts.loc[geo.index]

    return _to_long(pd.concat([geo, counts], axis=1), year)


era2_files = [
    (2017, "ukbusinessworkbook2017.xls"),
    (2018, "ukbusinessworkbook2018.xls"),
    (2019, "ukbusinessworkbook2019.xlsx"),
    (2020, "ukbusinessworkbook2020.xlsx"),
    (2021, "ukbusinessworkbook2021.xlsx"),
]

era2_dfs = {}
for year, fname in era2_files:
    p = HERE / fname
    if not p.exists():
        print(f"  SKIP: {fname}")
        continue
    df = read_era2(p, year)
    df = df[df["region_code"].apply(is_lad)]
    era2_dfs[year] = df
    print(f"  {year}: {df['region_code'].nunique()} LADs, {len(df):,} rows")

2022 to 2025



In [ ]:
def read_era3(path, year):
    raw = pd.read_excel(path, sheet_name="Table 1", header=None)
    sector_row = raw.iloc[3]

    sector_cols = [(c, parse_sic(sector_row.iloc[c]))
                   for c in range(1, raw.shape[1])
                   if parse_sic(sector_row.iloc[c])]

    parsed = raw.iloc[4:, 0].astype(str).str.extract(CODE_LABEL_RE)
    parsed.columns = ["region_code", "region_name"]
    keep = parsed.dropna(subset=["region_code"]).index

    counts = raw.iloc[4:, [c for c, _ in sector_cols]].copy()
    counts.columns = [sic for _, sic in sector_cols]
    counts = counts.loc[keep]

    return _to_long(pd.concat([parsed.loc[keep], counts], axis=1), year)


era3_files = [
    (2022, "ukbusinessworkbook2022.xlsx"),
    (2023, "ukbusinessworkbook2023.xlsx"),
    (2024, "ukbusinessworkbook2024.xlsx"),
    (2025, "ukbusinessworkbook2025new.xlsx"),
]

era3_dfs = {}
for year, fname in era3_files:
    p = HERE / fname
    if not p.exists():
        print(f"  SKIP: {fname}")
        continue
    df = read_era3(p, year)
    df = df[df["region_code"].apply(is_lad)]
    era3_dfs[year] = df
    print(f"  {year}: {df['region_code'].nunique()} LADs, {len(df):,} rows")

Combine everything

In [ ]:
all_dfs = {**era1_dfs, **era2_dfs, **era3_dfs}

if not all_dfs:
    print("No files loaded — check file paths.")
else:
    panel = pd.concat(all_dfs.values(), ignore_index=True).dropna(subset=["industry_group"])
    panel = panel.sort_values(["year","region_code","sic_codes"]).reset_index(drop=True)
    panel["area_type"] = "LAD"
    print(f"{len(panel):,} rows | {panel['region_code'].nunique()} unique LAD codes across all years")
    panel.head()

LAD coverage across years


In [ ]:
lad_counts = panel.groupby("year")["region_code"].nunique().rename("n_lads")
print(lad_counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(lad_counts.index, lad_counts.values, color="steelblue", width=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Distinct LAD codes")
ax.set_title("Number of LADs present per year")
ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
for yr, n in lad_counts.items():
    ax.text(yr, n + 1, str(n), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
years_present = sorted(panel["year"].unique())
lads_by_year  = {y: set(panel.loc[panel["year"]==y, "region_code"]) for y in years_present}

overlap_rows = []
for i, ya in enumerate(years_present):
    for yb in years_present[i+1:]:
        a, b = lads_by_year[ya], lads_by_year[yb]
        shared   = len(a & b)
        pct_of_a = shared / len(a) * 100
        pct_of_b = shared / len(b) * 100
        overlap_rows.append({
            "year_a": ya, "year_b": yb,
            "shared": shared,
            "pct_of_a": round(pct_of_a, 1),
            "pct_of_b": round(pct_of_b, 1),
            "min_pct":  round(min(pct_of_a, pct_of_b), 1),
        })

overlap = pd.DataFrame(overlap_rows)

# show the low-overlap pairs first — these are the problematic ones
overlap.sort_values("min_pct").head(20)

In [ ]:
# heatmap of pairwise overlap (min_pct)
import numpy as np

n = len(years_present)
mat = pd.DataFrame(np.nan, index=years_present, columns=years_present)
for _, row in overlap.iterrows():
    mat.loc[row.year_a, row.year_b] = row.min_pct
    mat.loc[row.year_b, row.year_a] = row.min_pct
np.fill_diagonal(mat.values, 100.0)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(mat.values, vmin=70, vmax=100, cmap="RdYlGn")
ax.set_xticks(range(n)); ax.set_xticklabels(years_present, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(years_present)
for i in range(n):
    for j in range(n):
        v = mat.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                    color="black" if v > 85 else "white")
plt.colorbar(im, ax=ax, label="% overlap (min of the two years)")
ax.set_title("LAD code overlap between year pairs (min %)")
plt.tight_layout()
plt.show()

In [ ]:
all_codes = panel[["region_code","region_name"]].drop_duplicates()
# how many years does each LAD code appear in?
years_per_lad = (
    panel.groupby("region_code")["year"]
    .nunique()
    .rename("n_years")
    .reset_index()
    .merge(all_codes, on="region_code", how="left")
    .sort_values("n_years")
)

print("Distribution of how many years each LAD code appears in:")
print(years_per_lad["n_years"].value_counts().sort_index().to_string())

In [ ]:
# LADs that don't span all available years
max_years = len(years_present)
partial = years_per_lad[years_per_lad["n_years"] < max_years].copy()

# which actual years does each partial LAD appear in?
partial_years = (
    panel[panel["region_code"].isin(partial["region_code"])]
    .groupby("region_code")["year"]
    .apply(sorted)
    .reset_index()
    .rename(columns={"year": "years_present"})
)

partial = partial.merge(partial_years, on="region_code")
print(f"{len(partial)} LAD codes don't appear in all {max_years} years")
partial.head(30)

In [ ]:
transitions = []
for ya, yb in zip(years_present[:-1], years_present[1:]):
    a, b = lads_by_year[ya], lads_by_year[yb]
    dropped = sorted(a - b)
    new     = sorted(b - a)
    transitions.append({
        "transition": f"{ya}→{yb}",
        "dropped": len(dropped),
        "new":     len(new),
        "net":     len(new) - len(dropped),
        "dropped_codes": dropped,
        "new_codes": new,
    })

tr = pd.DataFrame(transitions)
print(tr[["transition","dropped","new","net"]].to_string(index=False))

In [ ]:
# look at the biggest transitions in detail
for _, row in tr[tr["dropped"] > 2].iterrows():
    print(f"\n--- {row.transition} ---")
    if row.dropped_codes:
        dropped_names = (
            years_per_lad.set_index("region_code")["region_name"]
            .reindex(row.dropped_codes).fillna("?").to_dict()
        )
        print("  Dropped:", {k: v for k, v in dropped_names.items()})
    if row.new_codes:
        new_names = (
            years_per_lad.set_index("region_code")["region_name"]
            .reindex(row.new_codes).fillna("?").to_dict()
        )
        print("  New:    ", {k: v for k, v in new_names.items()})

In [ ]:
BREAK_THRESHOLD = 5  # >5 codes changing = boundary break

tr["is_break"] = (tr["dropped"] > BREAK_THRESHOLD) | (tr["new"] > BREAK_THRESHOLD)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

colors = ["firebrick" if b else "steelblue" for b in tr["is_break"]]

axes[0].bar(tr["transition"], tr["dropped"], color=colors)
axes[0].set_ylabel("Codes dropped")
axes[0].axhline(BREAK_THRESHOLD, color="k", lw=0.8, ls="--")

axes[1].bar(tr["transition"], tr["new"], color=colors)
axes[1].set_ylabel("Codes added")
axes[1].axhline(BREAK_THRESHOLD, color="k", lw=0.8, ls="--")
axes[1].tick_params(axis="x", rotation=45)

axes[0].set_title("Year-on-year LAD code churn  (red = likely boundary change)")
plt.tight_layout()
plt.show()

print("\nBreak years (do not span these without a concordance):")
for _, row in tr[tr["is_break"]].iterrows():
    print(f"  {row.transition}: {row.dropped} dropped, {row.new} added")

In [ ]:
# Which contiguous runs of years are internally consistent?
break_after = set(tr.loc[tr["is_break"], "transition"].str.split("→").str[0].astype(int))

safe_spans = []
current_span = [years_present[0]]
for yr in years_present[1:]:
    if (yr - 1) in break_after:
        safe_spans.append(current_span)
        current_span = [yr]
    else:
        current_span.append(yr)
safe_spans.append(current_span)

print("Safe year spans (stable LAD universe within each group):")
for span in safe_spans:
    codes_in_span = set.intersection(*[lads_by_year[y] for y in span])
    print(f"  {span[0]}–{span[-1]}  ({len(codes_in_span)} LADs consistently present)")

Consistent LADs only


In [ ]:
# take the longest safe span
best_span  = max(safe_spans, key=len)
stable_lads = set.intersection(*[lads_by_year[y] for y in best_span])

panel_clean = (
    panel[
        panel["year"].isin(best_span) &
        panel["region_code"].isin(stable_lads)
    ]
    .sort_values(["year","region_code","sic_codes"])
    .reset_index(drop=True)
)

print(f"Span: {best_span[0]}–{best_span[-1]}")
print(f"LADs: {panel_clean['region_code'].nunique()}")
print(f"Rows: {len(panel_clean):,}")
panel_clean.head()

In [ ]:
# verify the panel is actually balanced
expected = panel_clean["region_code"].nunique() * panel_clean["sic_codes"].nunique() * len(best_span)
print(f"Expected rows (balanced): {expected:,}")
print(f"Actual rows:              {len(panel_clean):,}")
print(f"Gap: {expected - len(panel_clean):,}  (should be 0 or very small)")

In [ ]:
out = HERE / "ons_panel_lad_clean.csv"
panel_clean.to_csv(out, index=False)
print(f"Saved: {out}")